# BASE LINEAR REGRESSION BENCHMARK

This notebook contains the basic data loading setup for Linear Regression benchmarking using **Polars** for optimized performance

**Datasets:**
- train_full: 5,337,414 rows, 94 columns
- test_full: 1,447,107 rows, 92 columns

**Purpose:**
Basic data loading and preparation for Linear Regression benchmarking

**Polars Benefits:**
- Faster data loading and processing
- Memory efficient operations
- Lazy evaluation capabilities
- Better multi-threading performance

## 1.1 IMPORTS AND CONFIGURATION

Setting up the environment with all necessary libraries and Polars configuration for optimal performance.

In [2]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import polars as pl
from pathlib import Path
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score
import time
from collections import Counter

# Polars configuration
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

# Set up paths
base_dir = Path("..")
full_train_path = base_dir / "data" / "train.parquet"
full_test_path = base_dir / "data" / "test.parquet"

print("✅ Imports and configuration complete")

✅ Imports and configuration complete


## 1.2 DATA LOADING

Loading the full train and test datasets using Polars with performance timing.

In [3]:
# ============================================
# LOAD FULL DATASETS
# ============================================
print("="*60)
print("LOADING FULL DATASETS")
print("="*60)

# Load full datasets using Polars
start_time = time.time()
train_full = pl.read_parquet(full_train_path)
test_full = pl.read_parquet(full_test_path)
load_time = time.time() - start_time

print(f"Train full: {train_full.shape}")
print(f"Test full: {test_full.shape}")
print(f"Load time: {load_time:.2f} seconds")

print(f"\n✅ Full datasets loaded")
print(f"   Train: {train_full.shape}")
print(f"   Test: {test_full.shape}")

LOADING FULL DATASETS
Train full: (5337414, 94)
Test full: (1447107, 92)
Load time: 2.90 seconds

✅ Full datasets loaded
   Train: (5337414, 94)
   Test: (1447107, 92)


## 2.0 BASIC DATA STRUCTURE ANALYSIS

Quick overview of dataset structure and basic information.

In [4]:
# ============================================
# BASIC DATA STRUCTURE ANALYSIS
# ============================================
print("="*60)
print("DATA STRUCTURE ANALYSIS")
print("="*60)

# Train dataset structure
print("\n TRAIN DATASET STRUCTURE:")
print(f"Shape: {train_full.shape}")
print(f"Columns: {train_full.columns}")
print(f"\nData types:")
# Correct way to get dtype counts in Polars
train_dtype_counts = Counter(str(dtype) for dtype in train_full.dtypes)
for dtype, count in train_dtype_counts.items():
    print(f"{dtype}: {count}")

# Test dataset structure  
print("\n TEST DATASET STRUCTURE:")
print(f"Shape: {test_full.shape}")
print(f"Columns: {test_full.columns}")
print(f"\nData types:")
# Correct way to get dtype counts in Polars
test_dtype_counts = Counter(str(dtype) for dtype in test_full.dtypes)
for dtype, count in test_dtype_counts.items():
    print(f"{dtype}: {count}")

# Column differences
train_cols = set(train_full.columns)
test_cols = set(test_full.columns)
only_in_train = train_cols - test_cols
only_in_test = test_cols - train_cols

print(f"\n COLUMN DIFFERENCES:")
print(f"Only in train: {only_in_train}")
print(f"Only in test: {only_in_test}")
print(f"Common columns: {len(train_cols & test_cols)}")

DATA STRUCTURE ANALYSIS

 TRAIN DATASET STRUCTURE:
Shape: (5337414, 94)
Columns: ['id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e', 'feature_f', 'feature_g', 'feature_h', 'feature_i', 'feature_j', 'feature_k', 'feature_l', 'feature_m', 'feature_n', 'feature_o', 'feature_p', 'feature_q', 'feature_r', 'feature_s', 'feature_t', 'feature_u', 'feature_v', 'feature_w', 'feature_x', 'feature_y', 'feature_z', 'feature_aa', 'feature_ab', 'feature_ac', 'feature_ad', 'feature_ae', 'feature_af', 'feature_ag', 'feature_ah', 'feature_ai', 'feature_aj', 'feature_ak', 'feature_al', 'feature_am', 'feature_an', 'feature_ao', 'feature_ap', 'feature_aq', 'feature_ar', 'feature_as', 'feature_at', 'feature_au', 'feature_av', 'feature_aw', 'feature_ax', 'feature_ay', 'feature_az', 'feature_ba', 'feature_bb', 'feature_bc', 'feature_bd', 'feature_be', 'feature_bf', 'feature_bg', 'feature_bh', 'feature_bi', 'feature_bj', 'feature_bk

In [7]:
# ============================================
# SEGMENT 1: BENCHMARK LINEAR REGRESSION (FILLNA 0)
# Score referencyjny: ~0.0626
# ============================================

import numpy as np
import polars as pl
from pathlib import Path
from sklearn.linear_model import LinearRegression
import time

# Metryka
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

# Wczytanie danych (twój kod)
base_dir = Path("..")
full_train_path = base_dir / "data" / "train.parquet"

print("="*60)
print("LOADING FULL DATASETS")
print("="*60)

start_time = time.time()
train_full = pl.read_parquet(full_train_path)
load_time = time.time() - start_time

print(f"Train full: {train_full.shape}")
print(f"Load time: {load_time:.2f} seconds")
print(f"\n✅ Dataset loaded")

# Feature selection
feature_cols = [col for col in train_full.columns if col.startswith('feature_')]
X = train_full.select(feature_cols).fill_null(0).to_numpy()
y = train_full.select("y_target").to_numpy().ravel()
w = train_full.select("weight").to_numpy().ravel()

# Model
model = LinearRegression(fit_intercept=True)
model.fit(X, y, sample_weight=w)

# Predykcja i score
y_pred = model.predict(X)
score = weighted_rmse_score(y, y_pred, w)

print(f"\nBenchmark Score: {score:.6f}")

LOADING FULL DATASETS
Train full: (5337414, 94)
Load time: 12.63 seconds

✅ Dataset loaded

Benchmark Score: 0.064388


In [5]:
def get_size_mb(df):
    return df.estimated_size() / 1024**2

print("Train size MB:", get_size_mb(train_full))
print("Test size MB:", get_size_mb(test_full))

Train size MB: 3940.168345451355
Test size MB: 1045.2955884933472


In [8]:
# ============================================
# SEGMENT 2: BENCHMARK XGBOOST (FLOAT64)
# ============================================

print("="*60)
print("BENCHMARK XGBOOST (FLOAT64)")
print("="*60)

import xgboost as xgb
import time

# Feature selection (te same co LR)
feature_cols = [col for col in train_full.columns if col.startswith('feature_')]
X = train_full.select(feature_cols).fill_null(0).to_numpy()
y = train_full.select("y_target").to_numpy().ravel()
w = train_full.select("weight").to_numpy().ravel()

# Konwersja do Float64 (wymuszona)
X_float64 = X.astype(np.float64)
print(f"X dtype: {X.dtype} → {X_float64.dtype}")

# XGBoost model
start_time = time.time()
xgb_model = xgb.XGBRegressor(
    tree_method='hist',
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    objective='reg:squarederror',
    random_state=42
)

xgb_model.fit(X_float64, y, sample_weight=w)
xgb_time = time.time() - start_time

# Predykcja i score
y_pred_xgb = xgb_model.predict(X_float64)
xgb_score = weighted_rmse_score(y, y_pred_xgb, w)

print(f"XGBoost training time: {xgb_time:.2f} seconds")
print(f"XGBoost Score: {xgb_score:.6f}")

BENCHMARK XGBOOST (FLOAT64)
X dtype: float64 → float64
XGBoost training time: 72.76 seconds
XGBoost Score: 0.182303


In [9]:
# ============================================
# SEGMENT 3: BENCHMARK LIGHTGBM (FLOAT64)
# ============================================

print("="*60)
print("BENCHMARK LIGHTGBM (FLOAT64)")
print("="*60)

import lightgbm as lgb
import time

# Feature selection (te same co LR)
feature_cols = [col for col in train_full.columns if col.startswith('feature_')]
X = train_full.select(feature_cols).fill_null(0).to_numpy()
y = train_full.select("y_target").to_numpy().ravel()
w = train_full.select("weight").to_numpy().ravel()

# Konwersja do Float64 (wymuszona)
X_float64 = X.astype(np.float64)
print(f"X dtype: {X.dtype} → {X_float64.dtype}")

# LightGBM model
start_time = time.time()
lgb_model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    num_leaves=31,
    learning_rate=0.1,
    n_estimators=100,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_float64, y, sample_weight=w)
lgb_time = time.time() - start_time

# Predykcja i score
y_pred_lgb = lgb_model.predict(X_float64)
lgb_score = weighted_rmse_score(y, y_pred_lgb, w)

print(f"LightGBM training time: {lgb_time:.2f} seconds")
print(f"LightGBM Score: {lgb_score:.6f}")

BENCHMARK LIGHTGBM (FLOAT64)
X dtype: float64 → float64
LightGBM training time: 63.88 seconds
LightGBM Score: 0.216946
